# FD-24 Сессия 1: Эксперимент E2 — Сравнение смесей данных (Слоты 1–4)
- **Модель:** `microsoft/unixcoder-base` @1024, `doc_budget: 384`
- **Окружение:** Kaggle 2×T4 GPU, PyTorch DDP, FP16
- **Слоты:**
  - Слот 1: E2(a) `real_only` (2 393 train)
  - Слот 2: E2(b) `synth_only` (19 008 train, только AST без LLM)
  - Слот 3: E2(c) `real_ast` (21 401 train)
  - Слот 4: E2(d) `full` (21 997 train, включая LLM)
- После каждого слота запускается оценка на `val`, `test_real` и `gold` для фиксации эффекта сдвига распределений.

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git checkout feat/FD-24-kaggle-experiments 2>/dev/null || !git checkout main 2>/dev/null || true
!git pull --ff-only || echo "PULL SKIPPED: убедись, что код свежий"

import os
os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!nvidia-smi

In [ ]:
import os, glob
import pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df

In [ ]:
# Генерация бейзлайнов M0 и M1 (подготовка для evaluate и стэкинга)
!python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
  --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
  --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json

In [ ]:
# Предполётка: smoke-тест весов (granite 48M, UniXcoder, ModernBERT, GTE)
import gc, torch
from transformers import AutoModel, AutoTokenizer
for m in ["ibm-granite/granite-embedding-small-english-r2",
          "microsoft/unixcoder-base",
          "answerdotai/ModernBERT-base",
          "Alibaba-NLP/gte-modernbert-base"]:
    tok = AutoTokenizer.from_pretrained(m)
    mod = AutoModel.from_pretrained(m).to("cuda")
    out = mod(**tok("def test(): pass", return_tensors="pt").to("cuda"))
    print(f"Smoke OK: {m:45s} shape={tuple(out.last_hidden_state.shape)}")
    del mod, out, tok
    gc.collect()
    torch.cuda.empty_cache()
print("GPU freed.")


In [ ]:
# ==============================================================================
# Слот 1: E2(a) Только реальные данные (2 393 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix real_only \
  --output_dir /kaggle/working/checkpoints/slot1_real --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot1_real/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot1_real/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot1_real/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot1_real/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot1_real/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot1_real/metrics.json --out_table /kaggle/working/checkpoints/slot1_real/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot1_real/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot1_real/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 2: E2(b) Только синтетика AST (19 008 train, без LLM)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix synth_only \
  --output_dir /kaggle/working/checkpoints/slot2_synth --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot2_synth/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot2_synth/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot2_synth/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot2_synth/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot2_synth/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot2_synth/metrics.json --out_table /kaggle/working/checkpoints/slot2_synth/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot2_synth/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot2_synth/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 3: E2(c) Реальные данные + AST (21 401 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix real_ast \
  --output_dir /kaggle/working/checkpoints/slot3_real_ast --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot3_real_ast/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot3_real_ast/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot3_real_ast/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot3_real_ast/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot3_real_ast/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot3_real_ast/metrics.json --out_table /kaggle/working/checkpoints/slot3_real_ast/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot3_real_ast/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot3_real_ast/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 4: E2(d) Полная смесь (+ LLM-пары, 21 997 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot4_full --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot4_full/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot4_full/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot4_full/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot4_full/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot4_full/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot4_full/metrics.json --out_table /kaggle/working/checkpoints/slot4_full/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot4_full/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot4_full/table_e1.csv

### Сводка Сессии 1

In [ ]:
import pandas as pd
df_runs = pd.read_csv("reports/runs.csv")
print(df_runs.tail(4).to_string(index=False))